# Lesson 5: Graph Algorithms (BFS, DFS, Shortest Paths)

**Source Reference:** [Data Structures and Algorithms in Python (freeCodeCamp / Jovian)](https://www.youtube.com/watch?v=pkYVOmU3MgA)

## 1. Graph Representation
A graph consists of **Nodes** (vertices) and **Edges** (connections).
In interviews, graphs are almost always represented using an **Adjacency List**, which is much more memory-efficient ($O(V + E)$) than an Adjacency Matrix ($O(V^2)$) for sparse data.

### The Engineering Requirement
We need a reusable Python class that can model:
1. **Undirected Graphs:** (e.g., Facebook friends - mutual connection)
2. **Directed Graphs:** (e.g., Twitter followers - one-way connection)
3. **Weighted Graphs:** (e.g., Google Maps - edges have distance/time costs)

In [7]:
class Graph:
    """A comprehensive Adjacency List representation of a Graph."""
    def __init__(self, num_nodes, edges, directed=False, weighted=False):
        self.num_nodes = num_nodes
        self.directed = directed
        self.weighted = weighted
        
        # Initialize an empty list for every node
        self.data = [[] for _ in range(num_nodes)]
        self.weight = [[] for _ in range(num_nodes)]
        
        # Populate the adjacency list
        for edge in edges:
            if self.weighted:
                node1, node2, weight = edge
                self.data[node1].append(node2)
                self.weight[node1].append(weight)
                
                # If undirected, the connection goes both ways
                if not self.directed:
                    self.data[node2].append(node1)
                    self.weight[node2].append(weight)
            else:
                node1, node2 = edge
                self.data[node1].append(node2)
                
                if not self.directed:
                    self.data[node2].append(node1)
                    
    def __repr__(self):
        result = ""
        if self.weighted:
            for i, (nodes, weights) in enumerate(zip(self.data, self.weight)):
                result += f"{i}: {list(zip(nodes, weights))}\n"
        else:
            for i, nodes in enumerate(self.data):
                result += f"{i}: {nodes}\n"
        return result

# Execution Demo
# num_nodes = 5
# edges = [(0, 1), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (3, 4)]
# graph = Graph(num_nodes, edges)

## 2. Graph Traversals
Traversing a graph means visiting every connected node exactly once. Because graphs can have cycles (loops), we must track which nodes have been `visited` to prevent infinite loops.

### Core Mechanic: BFS (Breadth-First Search)
* **Strategy:** Visit nodes layer-by-layer. Explore all immediate neighbors before moving deeper.
* **Data Structure:** Uses a **Queue** (First-In, First-Out).
* **Use Case:** Finding the shortest path in an *unweighted* graph, social network degrees of separation.

### Core Mechanic: DFS (Depth-First Search)
* **Strategy:** Pick a path and go as deep as possible until you hit a dead end, then backtrack.
* **Data Structure:** Uses a **Stack** (Last-In, First-Out) or Recursion.
* **Use Case:** Cycle detection, maze solving, topological sorting.

### Complexity Analysis
* **Time Complexity:** $O(V + E)$ where $V$ is vertices/nodes and $E$ is edges. We look at every node and follow every edge exactly once.
* **Space Complexity:** $O(V)$ to store the `visited` array and the Queue/Stack.

In [8]:
def bfs(graph, source):
    """Breadth-First Search using a Queue."""
    visited = [False] * graph.num_nodes
    queue = []
    result = []
    
    # Initialize the source node
    visited[source] = True    
    queue.append(source)
    
    while len(queue) > 0:
        # Queue Behavior: Pop from the FRONT (Index 0)
        curr = queue.pop(0)
        result.append(curr)
        
        # Scan all immediate neighbors
        for neighbor in graph.data[curr]:
            if not visited[neighbor]:
                visited[neighbor] = True
                queue.append(neighbor)
                
    return result

def dfs(graph, source):
    """Depth-First Search using a Stack."""
    visited = [False] * graph.num_nodes
    stack = [source]
    result = []
    
    while len(stack) > 0:
        # Stack Behavior: Pop from the BACK
        curr = stack.pop()
        
        if not visited[curr]:
            result.append(curr)
            visited[curr] = True
            
            # Add all neighbors to the stack to explore deeply
            for neighbor in graph.data[curr]:
                stack.append(neighbor)
                
    return result

## 3. Shortest Paths in Weighted Graphs
BFS works for unweighted graphs, but if edges have different costs (weights), we need **Dijkstra’s Algorithm**.

### Core Mechanic: Dijkstra's Algorithm
1. **Initialize:** Assign a tentative distance of `infinity` to all nodes except the source node (distance `0`).
2. **Greedy Selection:** Pick the unvisited node with the absolute smallest tentative distance.
3. **Relaxation:** Look at all its neighbors. If (Distance to Current Node + Cost of Edge) is *less* than the neighbor's current tentative distance, update the neighbor with this new, faster shortcut.
4. **Lock it in:** Mark the current node as `visited`. A visited node's shortest path is permanently locked and never checked again.

### Complexity Analysis
* **Time Complexity (Basic Array):** $O(V^2)$. Scanning the entire distance array to find the minimum unvisited node takes $O(V)$ time, done $V$ times.
* **Time Complexity (Priority Queue / Heap):** $O((V + E) \log V)$. *Note: In production and FAANG interviews, the `pick_next_node` scan is optimized using Python's `heapq` module to instantly grab the smallest distance in $O(\log V)$ time.*

In [9]:
def pick_next_node(distance, visited):
    """Linear scan to find the unvisited node with the smallest distance."""
    min_distance = float('inf')
    min_node = None
    
    for node in range(len(distance)):
        if not visited[node] and distance[node] < min_distance:
            min_node = node
            min_distance = distance[node]
            
    return min_node

def update_distances(graph, current, distance, parent):
    """Checks neighbors to see if a shorter path exists via the current node."""
    neighbors = graph.data[current]
    weights = graph.weight[current]
    
    for i, neighbor in enumerate(neighbors):
        weight = weights[i]
        
        # Relaxation Step: Is this new path strictly faster than the old known path?
        if distance[current] + weight < distance[neighbor]:
            distance[neighbor] = distance[current] + weight
            parent[neighbor] = current # Keep track of routing

def shortest_path(graph, source, dest):
    """Dijkstra's Algorithm to find the exact shortest path cost."""
    visited = [False] * graph.num_nodes
    distance = [float('inf')] * graph.num_nodes
    parent = [None] * graph.num_nodes
    
    # 1. Initialize Source
    distance[source] = 0
    
    # Loop until the destination is locked in
    while not visited[dest]:
        # 2. Greedy Selection
        current = pick_next_node(distance, visited)
        
        if current is None:
            break # No reachable nodes left
            
        # 3. Relaxation Step
        update_distances(graph, current, distance, parent)
        
        # 4. Lock it in
        visited[current] = True
        
    return distance[dest], distance, parent

# Execution Demo
# num_nodes = 6
# edges = [(0, 1, 4), (0, 2, 2), (1, 2, 5), (1, 3, 10), (2, 4, 3), (4, 3, 4), (3, 5, 11)]
# weighted_graph = Graph(num_nodes, edges, directed=True, weighted=True)
# shortest_dist, all_distances, routing_parents = shortest_path(weighted_graph, 0, 5)